In [1]:
import librosa
import numpy as np
import pandas as pd
import os

In [ ]:
def fix_length_axis1(x, desired_len):
    current_len = x.shape[1]
    if current_len < desired_len:
        return np.pad(x, ((0, 0), (0, desired_len - current_len)), mode='constant')
    else:
        return x[:, :desired_len]
def preprocess_audio(file_path, sr=22050, n_mels=128, n_mfcc=13, db_threshold=-70, desired_T=128):
    y, sr = librosa.load(file_path, sr=sr)

    intervals = librosa.effects.split(y, top_db=-db_threshold)
    if len(intervals) == 0:
        return None
    y_trimmed = np.concatenate([y[start:end] for start, end in intervals])

    mel = librosa.feature.melspectrogram(y=y_trimmed, sr=sr, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    mfcc = librosa.feature.mfcc(S=log_mel, sr=sr, n_mfcc=n_mfcc)

    def normalize(x):
        return (x - np.mean(x, axis=1, keepdims=True)) / (np.std(x, axis=1, keepdims=True) + 1e-6)

    log_mel = normalize(log_mel)
    mfcc = normalize(mfcc)

    log_mel = fix_length_axis1(log_mel, desired_T)
    mfcc = fix_length_axis1(mfcc, desired_T)

    if mfcc.shape[0] < n_mels:
        mfcc = np.pad(mfcc, ((0, n_mels - mfcc.shape[0]), (0, 0)), mode='constant')

    features = np.stack([log_mel, mfcc], axis=0)
    return features


In [ ]:
def process_csv_and_audio(csv_path, audio_dir):

    df = pd.read_csv(csv_path)

    processed_features = []
    for idx, row in df.iterrows():
        filename = row['filename']
        category = row['category']
        file_path = os.path.join(audio_dir, filename)

        if not os.path.exists(file_path):
            print(f"Warning: File {file_path} not found. Skipping.")
            processed_features.append(None)
            continue

        features = preprocess_audio(file_path)

        if features is None:
            print(f"Warning: No valid signal in {filename}. Skipping.")
            processed_features.append(None)
        else:
            processed_features.append(features)

    df['features'] = processed_features

    df = df[df['features'].notnull()].reset_index(drop=True)

    df = df.drop(columns=['filename',"fold","target","esc10","src_file","take"])

    return df

In [4]:
csv_path="/kaggle/input/background-noise-samples/ESC-50-master/meta/esc50.csv"
audio_dir="/kaggle/input/background-noise-samples/ESC-50-master/audio"
df = process_csv_and_audio(csv_path, audio_dir)

In [ ]:
unique_labels = df['category'].unique()
label_to_num = {label: idx for idx, label in enumerate(unique_labels, start=0)}

df['category'] = df['category'].map(label_to_num)

print(df)

      category                                           features
0            0  [[[-1.3041242, -0.961108, 0.47092807, 1.825600...
1            1  [[[1.9010074, 0.87697244, -0.5295544, -0.52955...
2            2  [[[-1.375014, -0.53689, -0.5661672, -1.7566607...
3            2  [[[1.0387102, 1.1265217, 0.7995339, 0.85837686...
4            3  [[[-1.9270833, -1.8488846, -0.5339584, -0.2647...
...        ...                                                ...
1995        31  [[[-1.707284, -2.2379482, -1.2671787, -1.01759...
1996         2  [[[2.3734825, 1.3057317, -0.5614439, 0.1241249...
1997        20  [[[0.48588324, 0.13742876, -0.3355257, 1.52704...
1998        14  [[[-0.9366855, 0.05324784, -1.218963, -1.22947...
1999         0  [[[-0.02667797, -0.20416196, -0.86303484, -0.3...

[2000 rows x 2 columns]


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

2025-06-30 13:05:47.561921: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751288747.814115      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751288747.886262      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
from tensorflow.keras.callbacks import Callback

class StopAtValAccuracy(Callback):
    def __init__(self, target=0.76):
        super().__init__()
        self.target = target

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get("val_accuracy")
        if val_acc is not None and val_acc >= self.target:
            print(f"\n🎯 Reached {val_acc*100:.2f}% validation accuracy. Stopping training.")
            self.model.stop_training = True


In [ ]:
X = np.stack(df['features'].values).astype(np.float32)  
X = np.transpose(X, (0, 2, 3, 1))                        
y = df['category'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
def build_model(input_shape, num_classes):
    model = models.Sequential()
    
    model.add(layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(256, (3, 3), padding='same'))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.AveragePooling2D(pool_size=(2, 2)))

    model.add(layers.Conv2D(256, (3, 3), padding='same'))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.AveragePooling2D(pool_size=(2, 2)))

    model.add(layers.Conv2D(128, (3, 3), padding='same'))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.AveragePooling2D(pool_size=(2, 2)))

    model.add(layers.Conv2D(256, (3, 3), padding='same'))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(128, activation='tanh'))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

In [ ]:
input_shape = (128, 128, 2)
num_classes = len(np.unique(y))
optimizer = Adam(learning_rate=0.0005)
model = build_model(input_shape, num_classes)
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
stop_callback = StopAtValAccuracy(target=0.76)


history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[stop_callback]
)

2025-06-30 13:06:03.310562: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 163s 6s/step - accuracy: 0.1059 - loss: 3.5468 - val_accuracy: 0.0200 - val_loss: 3.8207
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 154s 6s/step - accuracy: 0.2608 - loss: 2.7021 - val_accuracy: 0.0225 - val_loss: 3.9125
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 177s 7s/step - accuracy: 0.3857 - loss: 2.3716 - val_accuracy: 0.0250 - val_loss: 3.9004
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 179s 6s/step - accuracy: 0.4262 - loss: 2.1946 - val_accuracy: 0.0475 - val_loss: 3.8789
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 157s 6s/step - accuracy: 0.4729 - loss: 1.9797 - val_accuracy: 0.0775 - val_loss: 3.8799
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 154s 6s/step - accuracy: 0.5029 - loss: 1.8484 - val_accuracy: 0.0725 - val_loss: 3.6294
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 182s 7s/step - accuracy: 0.5545 - loss: 1.7296 - val_accuracy: 0.0700 - val_loss: 3.7188
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 154s 6s/step - accuracy: 0.5932 - loss: 1.5821 - val_accuracy: 0.

In [11]:
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 861ms/step - accuracy: 0.7740 - loss: 0.7850
Validation Accuracy: 76.75%
